In [1]:
import librosa
import librosa.display
import numpy as np
import pandas as pd
import os
from typing import List, Tuple, Dict
import matplotlib.pyplot as plt
import random
import cv2
# from vgg16 import VGG16_LSTM
import pickle
import torch.nn as nn
import torch
import datetime
import torch.nn.functional as F
import torchvision.models as models
from torchvision import transforms, utils, models
from torch.utils.data import Dataset, DataLoader
from facenet_pytorch import MTCNN
import logging
import torch
import torch.nn as nn
from vivit_ibca_probe import ViViTBackbone, ViViTIBCALinearProbe
import gc
from einops import rearrange
from tqdm import tqdm
# from kw_r2plus1d_video_swin_transformer import SwinTransformer3D
batchsz = 4
frame_nums=15
start_epoch = 0
test_flage=False
model_name ="r2plus1d+ViViT + AST fusion after transformer blocks"
logger = logging.getLogger(__name__)
logger.setLevel(logging.DEBUG)
formatter = logging.Formatter('[%(asctime)s]::%(module)s::%(levelname)s::%(message)s')
streamHandler = logging.StreamHandler()
streamHandler.setFormatter(formatter)
fileHandler = logging.FileHandler('./LOG/personalityLog.log')
fileHandler.setFormatter(formatter)
logger.addHandler(streamHandler)
logger.addHandler(fileHandler)
USE_CUDA = torch.cuda.is_available()
# device = torch.device('cuda:1' if USE_CUDA else 'cpu')

In [2]:
logger.debug('============={}+2D Video Vision Transformer Strat!============='.format(model_name))
USE_CUDA = torch.cuda.is_available()
print(USE_CUDA)

device = torch.device('cuda:1' if USE_CUDA else 'cpu')
print('학습을 진행하는 기기:', device)

[2026-09-25 01:46:00,048]::2549926179::DEBUG::=============r2plus1d+ViViT + AST fusion after transformer blocks+2D Video Vision Transformer Strat!=============


True
학습을 진행하는 기기: cuda:1


In [3]:
def getdata(pathname,datatype):
    if datatype=="train":
        final_data_set=[]
        output_file=open(pathname, "rb")
        for i in range(1):
            final_data_set.extend(pickle.load(output_file))
        return final_data_set
    elif datatype=="valid":
        final_data_set=[]
        output_file=open(pathname, "rb")
        for i in range(1):
            valid_data_set=[]
            final_data_set.extend(pickle.load(output_file))
            del valid_data_set
        return final_data_set
print(1)

1


In [4]:

save_model_file_path = './save_swintransformer_folder/{}_{}.{}'
print(1)

1


In [5]:
train_set_data=getdata("/data1/qx/mean_based_arg/udiva/multimodal_fusion/data/speaking_only/train_set.dat","train")
print(1)
validation_set_data=getdata("/data1/qx/mean_based_arg/udiva/multimodal_fusion/data/speaking_only/valid_set.dat","valid")
print(1)

1
1


In [6]:
transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.ToTensor(),
])
print(1)

1


In [7]:
class ChalearnDataset(Dataset):
    def __init__(self,audiodata,fullshot,tagdata,transform=None):
        self.audiodata=audiodata
        self.fullshot=fullshot
        self.tagdata=tagdata
        self.transform = transform  # 표준화 여부
    def __len__(self):
        return len(self.fullshot)

    def __getitem__(self, idx):
        if torch.is_tensor(idx):
            idx = idx.tolist()#텐서의 경우 목록으로 돌아가기
        audiodata = self.audiodata[idx]
        audiodata = torch.FloatTensor(audiodata)
        fullshot=self.fullshot[idx]
        fullshot=torch.FloatTensor(fullshot)
        big_five_sorces=self.tagdata[idx]
        big_five_sorces = torch.FloatTensor(big_five_sorces)
        return audiodata,fullshot,big_five_sorces
print(1)

1


In [8]:
num_workerssz = 20
lr = 3e-05
epochs = 120
print(1)
# Multi-seed setting
SEEDS = [42, 123, 777, 1004, 2024]
patience = 20

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)


1


In [9]:
train_set_data = ChalearnDataset(audiodata=train_set_data[0],fullshot = train_set_data[1],tagdata=train_set_data[2],transform=transform)
val_set_data = ChalearnDataset(audiodata=validation_set_data[0],fullshot = validation_set_data[1],tagdata=validation_set_data[2],transform=transform)
print(1)


1


In [10]:
def extract_ocean(data,batch_size):
    o=[]
    c=[]
    e=[]
    a=[]
    n=[]
    if batch_size >1:
        for i in range(batch_size):
            o.append(data[i][0].item())
            c.append(data[i][1].item())
            e.append(data[i][2].item())
            a.append(data[i][3].item())
            n.append(data[i][4].item())
        o = torch.tensor(o) 
        c = torch.tensor(c) 
        e = torch.tensor(e) 
        a = torch.tensor(a) 
        n = torch.tensor(n) 
    elif batch_size == 1:
        data=data.squeeze()
        for i in range(batch_size):
            o.append(data[0].item())
            c.append(data[1].item())
            e.append(data[2].item())
            a.append(data[3].item())
            n.append(data[4].item())
            
        o = torch.tensor(o) 
        c = torch.tensor(c) 
        e = torch.tensor(e) 
        a = torch.tensor(a) 
        n = torch.tensor(n)
    return o,c,e,a,n

In [11]:
# ============================================================
# ViViT-IBCA Block-wise Linear Probe -- UDIVA / MSE
# ============================================================
# Frozen backbone; only independent linear probe heads are trained.
#
# IMPORTANT:
#   Metric = MSE (lower is better).
#   No 1-MSE is calculated or reported.
#   O/C/E/A/N are also reported directly as MSE.
# ============================================================

PROBE_SEED = 42
probe_lr = 1e-3
probe_weight_decay = 1e-4
probe_epochs = 50
patience = 10

BACKBONE_CHECKPOINT = "./save_swintransformer_folder/model_best_multiseed.pth"

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % (2 ** 32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)

set_seed(PROBE_SEED)

g = torch.Generator()
g.manual_seed(PROBE_SEED)

train_dataloader = DataLoader(
    dataset=train_set_data,
    batch_size=batchsz,
    shuffle=True,
    num_workers=num_workerssz,
    worker_init_fn=seed_worker,
    generator=g
)

val_dataloader = DataLoader(
    dataset=val_set_data,
    batch_size=batchsz,
    shuffle=False,
    num_workers=num_workerssz,
    worker_init_fn=seed_worker
)

# Same ViViT-IBCA backbone configuration as the original notebook.
backbone = ViViTBackbone(
    2, 14, 14,
    2, 2, 2,
    5,
    96,
    6,
    8,
    384,
    model=3
).to(device)

ckpt = torch.load(BACKBONE_CHECKPOINT, map_location=device)
state_dict = ckpt["model"] if isinstance(ckpt, dict) and "model" in ckpt else ckpt
backbone.load_state_dict(state_dict, strict=True)

model = ViViTIBCALinearProbe(
    backbone=backbone,
    dim=96,
    num_outputs=5
).to(device)

assert sum(p.numel() for p in model.backbone.parameters() if p.requires_grad) == 0
print("Backbone trainable parameters: 0")
print("Probe trainable parameters:",
      sum(p.numel() for p in model.probe_heads.parameters() if p.requires_grad))
print(f"Metric: MSE | Patience: {patience}")

criterion = nn.MSELoss().to(device)
optimizer = torch.optim.AdamW(
    model.probe_heads.parameters(),
    lr=probe_lr,
    weight_decay=probe_weight_decay
)

probe_names = list(model.probe_heads.keys())
best_mse = {name: float("inf") for name in probe_names}
best_epoch = {name: -1 for name in probe_names}
best_mean_val_mse = float("inf")
no_improve_count = 0

for epoch in range(probe_epochs):
    # -------------------- Train probes only --------------------
    model.train()
    train_sq_sum = {name: 0.0 for name in probe_names}
    train_count = 0

    for audiodata, fullshot, big_five_data in tqdm(
        train_dataloader, desc=f"Probe Train {epoch+1}/{probe_epochs}"
    ):
        fullshot = fullshot.permute(0, 4, 1, 2, 3).to(device)
        big_five_data = big_five_data.permute(0, 2, 1).squeeze().to(device)
        if big_five_data.ndim == 1:
            big_five_data = big_five_data.unsqueeze(0)
        audiodata = audiodata.to(device)

        optimizer.zero_grad()
        outputs = model(fullshot, audiodata)

        losses = {
            name: criterion(outputs[name], big_five_data)
            for name in probe_names
        }
        sum(losses.values()).backward()
        optimizer.step()

        train_count += big_five_data.numel()
        for name in probe_names:
            train_sq_sum[name] += torch.square(
                outputs[name].detach() - big_five_data
            ).sum().item()

    train_mse = {
        name: train_sq_sum[name] / train_count
        for name in probe_names
    }

    # -------------------- Validation --------------------
    model.eval()
    val_sq_sum = {name: 0.0 for name in probe_names}
    val_trait_sq_sum = {
        name: np.zeros(5, dtype=np.float64)
        for name in probe_names
    }
    val_count = 0
    val_sample_count = 0

    with torch.no_grad():
        for audiodata, fullshot, big_five_data in tqdm(
            val_dataloader, desc=f"Probe Valid {epoch+1}/{probe_epochs}"
        ):
            fullshot = fullshot.permute(0, 4, 1, 2, 3).to(device)
            big_five_data = big_five_data.permute(0, 2, 1).squeeze().to(device)
            if big_five_data.ndim == 1:
                big_five_data = big_five_data.unsqueeze(0)
            audiodata = audiodata.to(device)

            outputs = model(fullshot, audiodata)
            val_count += big_five_data.numel()
            val_sample_count += big_five_data.shape[0]

            for name in probe_names:
                sq_err = torch.square(outputs[name] - big_five_data)
                val_sq_sum[name] += sq_err.sum().item()
                val_trait_sq_sum[name] += (
                    sq_err.sum(dim=0).detach().cpu().numpy()
                )

    val_mse = {
        name: val_sq_sum[name] / val_count
        for name in probe_names
    }
    val_trait_mse = {
        name: val_trait_sq_sum[name] / val_sample_count
        for name in probe_names
    }

    for name in probe_names:
        if val_mse[name] < best_mse[name]:
            best_mse[name] = val_mse[name]
            best_epoch[name] = epoch + 1

    mean_val_mse = float(np.mean([val_mse[n] for n in probe_names]))

    if mean_val_mse < best_mean_val_mse:
        best_mean_val_mse = mean_val_mse
        no_improve_count = 0
    else:
        no_improve_count += 1

    print("\n" + "-" * 132)
    print(
        f"Epoch {epoch+1:03d} | "
        f"Mean Val MSE={mean_val_mse:.6f} | "
        f"Best Mean Val MSE={best_mean_val_mse:.6f} | "
        f"Patience={no_improve_count}/{patience}"
    )

    for name in probe_names:
        o_mse, c_mse, e_mse, a_mse, n_mse = val_trait_mse[name]
        print(
            f"{name:<10} | "
            f"Train MSE {train_mse[name]:.6f} | "
            f"Val MSE {val_mse[name]:.6f} | "
            f"Best MSE {best_mse[name]:.6f} | "
            f"O MSE {o_mse:.6f} | C MSE {c_mse:.6f} | "
            f"E MSE {e_mse:.6f} | A MSE {a_mse:.6f} | "
            f"N MSE {n_mse:.6f}"
        )

    if no_improve_count >= patience:
        print(
            f"\nEarly stopping at epoch {epoch+1} | "
            f"Best Mean Val MSE={best_mean_val_mse:.6f} | "
            f"Patience={no_improve_count}/{patience}"
        )
        break

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\n" + "=" * 72)
print("FINAL ViViT-IBCA LINEAR PROBE RESULTS -- UDIVA / MSE")
print("=" * 72)
print(f"{'Representation':<18}{'Best Epoch':>12}{'Best MSE':>16}")
print("-" * 72)
for name in probe_names:
    print(
        f"{name:<18}{best_epoch[name]:>12d}{best_mse[name]:>16.6f}"
    )
print("=" * 72)

del model, backbone, optimizer, train_dataloader, val_dataloader
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


---------------AST Model Summary---------------
ImageNet pretraining: True, AudioSet pretraining: True
frequncey stride=10, time stride=10
number of patches=299
Backbone trainable parameters: 0
Probe trainable parameters: 2910
Metric: MSE | Patience: 10


Probe Valid 1/50: 100%|██████████| 191/191 [00:36<00:00,  5.24it/s]



------------------------------------------------------------------------------------------------------------------------------------
Epoch 001 | Mean Val MSE=1.352391 | Best Mean Val MSE=1.352391 | Patience=0/10
block1     | Train MSE 0.133238 | Val MSE 1.351090 | Best MSE 1.351090 | O MSE 1.037099 | C MSE 1.304098 | E MSE 1.315030 | A MSE 1.662358 | N MSE 1.436864
block2     | Train MSE 0.101734 | Val MSE 1.345171 | Best MSE 1.345171 | O MSE 1.023629 | C MSE 1.282903 | E MSE 1.324821 | A MSE 1.659877 | N MSE 1.434623
block3     | Train MSE 0.109215 | Val MSE 1.345804 | Best MSE 1.345804 | O MSE 1.018648 | C MSE 1.302998 | E MSE 1.308479 | A MSE 1.658829 | N MSE 1.440065
block4     | Train MSE 0.105902 | Val MSE 1.341971 | Best MSE 1.341971 | O MSE 1.037957 | C MSE 1.331318 | E MSE 1.258480 | A MSE 1.643366 | N MSE 1.438735
block5     | Train MSE 0.102955 | Val MSE 1.359710 | Best MSE 1.359710 | O MSE 1.053607 | C MSE 1.347598 | E MSE 1.271149 | A MSE 1.657775 | N MSE 1.468420
block6 

Probe Valid 2/50: 100%|██████████| 191/191 [00:36<00:00,  5.29it/s]



------------------------------------------------------------------------------------------------------------------------------------
Epoch 002 | Mean Val MSE=1.354037 | Best Mean Val MSE=1.352391 | Patience=1/10
block1     | Train MSE 0.048472 | Val MSE 1.344678 | Best MSE 1.344678 | O MSE 1.043916 | C MSE 1.323036 | E MSE 1.291150 | A MSE 1.664329 | N MSE 1.400959
block2     | Train MSE 0.041220 | Val MSE 1.345629 | Best MSE 1.345171 | O MSE 1.041216 | C MSE 1.317156 | E MSE 1.302532 | A MSE 1.662358 | N MSE 1.404880
block3     | Train MSE 0.035769 | Val MSE 1.349276 | Best MSE 1.345804 | O MSE 1.032553 | C MSE 1.347535 | E MSE 1.318231 | A MSE 1.658101 | N MSE 1.389960
block4     | Train MSE 0.029741 | Val MSE 1.347186 | Best MSE 1.341971 | O MSE 1.051484 | C MSE 1.364689 | E MSE 1.293515 | A MSE 1.642454 | N MSE 1.383788
block5     | Train MSE 0.024921 | Val MSE 1.364343 | Best MSE 1.359710 | O MSE 1.057696 | C MSE 1.384466 | E MSE 1.325458 | A MSE 1.659023 | N MSE 1.395070
block6 

...


------------------------------------------------------------------------------------------------------------------------------------
Epoch 013 | Mean Val MSE=1.345921 | Best Mean Val MSE=1.331853 | Patience=8/10
block1     | Train MSE 0.033256 | Val MSE 1.325409 | Best MSE 1.322125 | O MSE 1.068455 | C MSE 1.269729 | E MSE 1.282899 | A MSE 1.633276 | N MSE 1.372685
block2     | Train MSE 0.027000 | Val MSE 1.338520 | Best MSE 1.323677 | O MSE 1.049404 | C MSE 1.279699 | E MSE 1.302316 | A MSE 1.663708 | N MSE 1.397473
block3     | Train MSE 0.022156 | Val MSE 1.340137 | Best MSE 1.321777 | O MSE 1.031691 | C MSE 1.284417 | E MSE 1.290801 | A MSE 1.668991 | N MSE 1.424788
block4     | Train MSE 0.018990 | Val MSE 1.348297 | Best MSE 1.319187 | O MSE 1.034157 | C MSE 1.325224 | E MSE 1.277180 | A MSE 1.635436 | N MSE 1.469487
block5     | Train MSE 0.016821 | Val MSE 1.358028 | Best MSE 1.336480 | O MSE 1.039511 | C MSE 1.386573 | E MSE 1.289598 | A MSE 1.640603 | N MSE 1.433857
block6 

Probe Valid 14/50: 100%|██████████| 191/191 [00:31<00:00,  5.97it/s]



------------------------------------------------------------------------------------------------------------------------------------
Epoch 014 | Mean Val MSE=1.340328 | Best Mean Val MSE=1.331853 | Patience=9/10
block1     | Train MSE 0.032699 | Val MSE 1.322252 | Best MSE 1.322125 | O MSE 1.054674 | C MSE 1.278231 | E MSE 1.264275 | A MSE 1.637056 | N MSE 1.377025
block2     | Train MSE 0.026498 | Val MSE 1.337457 | Best MSE 1.323677 | O MSE 1.039131 | C MSE 1.307943 | E MSE 1.294669 | A MSE 1.655843 | N MSE 1.389697
block3     | Train MSE 0.021788 | Val MSE 1.336282 | Best MSE 1.321777 | O MSE 1.021490 | C MSE 1.309895 | E MSE 1.293756 | A MSE 1.651832 | N MSE 1.404437
block4     | Train MSE 0.018755 | Val MSE 1.339700 | Best MSE 1.319187 | O MSE 1.026131 | C MSE 1.343874 | E MSE 1.285585 | A MSE 1.611441 | N MSE 1.431470
block5     | Train MSE 0.016646 | Val MSE 1.351252 | Best MSE 1.336480 | O MSE 1.026732 | C MSE 1.422680 | E MSE 1.296956 | A MSE 1.614834 | N MSE 1.395060
block6 

Probe Valid 15/50: 100%|██████████| 191/191 [00:32<00:00,  5.87it/s]



------------------------------------------------------------------------------------------------------------------------------------
Epoch 015 | Mean Val MSE=1.338021 | Best Mean Val MSE=1.331853 | Patience=10/10
block1     | Train MSE 0.032248 | Val MSE 1.322515 | Best MSE 1.322125 | O MSE 1.061138 | C MSE 1.278888 | E MSE 1.254879 | A MSE 1.632932 | N MSE 1.384737
block2     | Train MSE 0.026070 | Val MSE 1.335936 | Best MSE 1.323677 | O MSE 1.048130 | C MSE 1.293571 | E MSE 1.284832 | A MSE 1.658614 | N MSE 1.394532
block3     | Train MSE 0.021399 | Val MSE 1.334420 | Best MSE 1.321777 | O MSE 1.032873 | C MSE 1.292859 | E MSE 1.265086 | A MSE 1.662815 | N MSE 1.418467
block4     | Train MSE 0.018441 | Val MSE 1.336323 | Best MSE 1.319187 | O MSE 1.034039 | C MSE 1.333438 | E MSE 1.247637 | A MSE 1.624888 | N MSE 1.441615
block5     | Train MSE 0.016447 | Val MSE 1.343811 | Best MSE 1.336480 | O MSE 1.033299 | C MSE 1.389852 | E MSE 1.249674 | A MSE 1.635662 | N MSE 1.410569
block6